In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    current_path = start_path.resolve()

    for candidate in [current_path, *current_path.parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR
from src.audio_features import extract_audio_features

PROCESSED_DIR = Path(PROCESSED_DATA_DIR)

SPLIT_FILE = (
    PROCESSED_DIR / "dataset_split.csv"
)

FEATURE_FILE = (
    PROCESSED_DIR / "audio_features.csv"
)

ERROR_FILE = (
    PROCESSED_DIR / "feature_extraction_errors.csv"
)

CHECKPOINT_FILE = (
    PROCESSED_DIR / "audio_features_checkpoint.csv"
)

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Dataset split exists:", SPLIT_FILE.exists())

if ".venv" not in sys.executable:
    raise RuntimeError(
        "Select the Music Genre Classification AI (.venv) kernel."
    )

if not SPLIT_FILE.exists():
    raise FileNotFoundError(
        "dataset_split.csv was not found. "
        "Run 03_dataset_split.ipynb first."
    )

Python: d:\Projects\music-genre-classification-AI\.venv\Scripts\python.exe
Project root: D:\Projects\music-genre-classification-AI
Dataset split exists: True


In [2]:
split_df = pd.read_csv(SPLIT_FILE)

required_columns = {
    "file_path",
    "filename",
    "genre",
    "split",
}

missing_columns = required_columns - set(split_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns: {sorted(missing_columns)}"
    )

split_df = (
    split_df
    .drop_duplicates(subset=["file_path"])
    .reset_index(drop=True)
)

print("Total tracks:", len(split_df))

display(
    split_df["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .rename("track_count")
    .to_frame()
)

display(split_df.head())

Total tracks: 999


,track_count
split,
train,699
validation,150
test,150


,file_path,filename,genre,sample_rate,channels,frames,duration_seconds,size_bytes,status,warnings,split
0,data\raw\genres_original\country\country.00061...,country.00061.wav,country,22050.0,1.0,661794.0,30.013333,1323632,valid,NaN,train
1,data\raw\genres_original\classical\classical.0...,classical.00046.wav,classical,22050.0,1.0,661760.0,30.011791,1323564,valid,NaN,validation
2,data\raw\genres_original\country\country.00005...,country.00005.wav,country,22050.0,1.0,666820.0,30.241270,1333684,valid,NaN,train
3,data\raw\genres_original\disco\disco.00049.wav,disco.00049.wav,disco,22050.0,1.0,661504.0,30.000181,1323052,valid,NaN,train
4,data\raw\genres_original\hiphop\hiphop.00022.wav,hiphop.00022.wav,hiphop,22050.0,1.0,661794.0,30.013333,1323632,valid,NaN,validation


In [3]:
sample_row = split_df.iloc[0]

sample_path = Path(
    sample_row["file_path"]
)

if not sample_path.is_absolute():
    sample_path = PROJECT_ROOT / sample_path

print("Testing file:", sample_path)
print("File exists:", sample_path.exists())
print("Genre:", sample_row["genre"])

sample_features = extract_audio_features(
    sample_path
)

print("Number of extracted features:", len(sample_features))

display(
    pd.Series(
        sample_features,
        name="value",
    ).to_frame().head(30)
)

Testing file: D:\Projects\music-genre-classification-AI\data\raw\genres_original\country\country.00061.wav
File exists: True
Genre: country
Number of extracted features: 90


,value
loaded_duration_seconds,30.000000
tempo_bpm,123.046875
mfcc_01_mean,-482.063802
mfcc_01_std,68.180736
mfcc_02_mean,135.860073
mfcc_02_std,28.998235
mfcc_03_mean,5.928498
mfcc_03_std,16.122102
mfcc_04_mean,45.860501
mfcc_04_std,11.991578


In [4]:
feature_records = []
error_records = []

total_tracks = len(split_df)

for row_index, row in split_df.iterrows():
    audio_path = Path(row["file_path"])

    if not audio_path.is_absolute():
        audio_path = PROJECT_ROOT / audio_path

    try:
        extracted_features = extract_audio_features(
            audio_path
        )

        feature_records.append(
            {
                "file_path": row["file_path"],
                "filename": row["filename"],
                "genre": row["genre"],
                "split": row["split"],
                **extracted_features,
            }
        )

    except Exception as error:
        error_records.append(
            {
                "file_path": row["file_path"],
                "filename": row["filename"],
                "genre": row["genre"],
                "split": row["split"],
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
        )

    completed = row_index + 1

    if completed % 25 == 0 or completed == total_tracks:
        print(
            f"Processed {completed}/{total_tracks} tracks "
            f"| Successful: {len(feature_records)} "
            f"| Errors: {len(error_records)}"
        )

    # Save a temporary checkpoint every 100 tracks
    if completed % 100 == 0:
        pd.DataFrame(feature_records).to_csv(
            CHECKPOINT_FILE,
            index=False,
        )

print("\nFeature extraction completed.")
print("Successful tracks:", len(feature_records))
print("Failed tracks:", len(error_records))

Processed 25/999 tracks | Successful: 25 | Errors: 0
Processed 50/999 tracks | Successful: 50 | Errors: 0
Processed 75/999 tracks | Successful: 75 | Errors: 0
Processed 100/999 tracks | Successful: 100 | Errors: 0
Processed 125/999 tracks | Successful: 125 | Errors: 0
Processed 150/999 tracks | Successful: 150 | Errors: 0
Processed 175/999 tracks | Successful: 175 | Errors: 0
Processed 200/999 tracks | Successful: 200 | Errors: 0
Processed 225/999 tracks | Successful: 225 | Errors: 0
Processed 250/999 tracks | Successful: 250 | Errors: 0
Processed 275/999 tracks | Successful: 275 | Errors: 0
Processed 300/999 tracks | Successful: 300 | Errors: 0
Processed 325/999 tracks | Successful: 325 | Errors: 0
Processed 350/999 tracks | Successful: 350 | Errors: 0
Processed 375/999 tracks | Successful: 375 | Errors: 0
Processed 400/999 tracks | Successful: 400 | Errors: 0
Processed 425/999 tracks | Successful: 425 | Errors: 0
Processed 450/999 tracks | Successful: 450 | Errors: 0
Processed 475/99

In [5]:
feature_df = pd.DataFrame(
    feature_records
)

error_df = pd.DataFrame(
    error_records
)

if feature_df.empty:
    raise RuntimeError(
        "No features were extracted."
    )

metadata_columns = [
    "file_path",
    "filename",
    "genre",
    "split",
]

feature_columns = [
    column
    for column in feature_df.columns
    if column not in metadata_columns
]

feature_df[feature_columns] = (
    feature_df[feature_columns]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

rows_with_missing_values = (
    feature_df[feature_columns]
    .isna()
    .any(axis=1)
)

print("Feature table shape:", feature_df.shape)
print("Number of numerical features:", len(feature_columns))
print(
    "Rows containing missing values:",
    int(rows_with_missing_values.sum()),
)

if rows_with_missing_values.any():
    display(
        feature_df.loc[
            rows_with_missing_values,
            metadata_columns,
        ]
    )

    feature_df = (
        feature_df.loc[
            ~rows_with_missing_values
        ]
        .reset_index(drop=True)
    )

display(feature_df.head())

Feature table shape: (999, 94)
Number of numerical features: 90
Rows containing missing values: 0


,file_path,filename,genre,split,loaded_duration_seconds,tempo_bpm,mfcc_01_mean,mfcc_01_std,mfcc_02_mean,mfcc_02_std,...,spectral_contrast_03_mean,spectral_contrast_03_std,spectral_contrast_04_mean,spectral_contrast_04_std,spectral_contrast_05_mean,spectral_contrast_05_std,spectral_contrast_06_mean,spectral_contrast_06_std,spectral_contrast_07_mean,spectral_contrast_07_std
0,data\raw\genres_original\country\country.00061...,country.00061.wav,country,train,30.0,123.046875,-482.063802,68.180736,135.860073,28.998235,...,23.540774,4.545851,21.173070,3.812318,21.600273,3.830388,19.884863,3.602342,37.326387,4.477069
1,data\raw\genres_original\classical\classical.0...,classical.00046.wav,classical,validation,30.0,129.199219,-373.008461,23.495534,118.423067,19.340320,...,18.463920,3.502241,19.849700,2.520823,20.713156,2.359407,20.558529,2.448312,21.254693,2.730834
2,data\raw\genres_original\country\country.00005...,country.00005.wav,country,train,30.0,129.199219,-393.541695,27.496700,62.388383,15.932810,...,19.019879,4.381183,19.061713,3.924187,19.638747,3.195603,17.685056,2.671622,17.485123,2.074316
3,data\raw\genres_original\disco\disco.00049.wav,disco.00049.wav,disco,train,30.0,123.046875,-437.577377,60.125883,98.671647,30.238519,...,17.128404,4.040533,16.994300,3.413414,18.271373,3.387091,18.764664,3.561410,42.711696,6.376284
4,data\raw\genres_original\hiphop\hiphop.00022.wav,hiphop.00022.wav,hiphop,validation,30.0,99.384014,-501.525654,73.550980,110.210732,29.239844,...,18.053938,4.430873,16.798288,3.913257,16.911758,3.270236,16.258567,2.861101,40.968474,4.152934


In [6]:
print("Usable feature rows:", len(feature_df))

split_summary = pd.crosstab(
    feature_df["genre"],
    feature_df["split"],
)

split_summary = split_summary.reindex(
    columns=["train", "validation", "test"],
    fill_value=0,
)

split_summary["total"] = (
    split_summary.sum(axis=1)
)

display(split_summary)

Usable feature rows: 999


split,train,validation,test,total
genre,,,,
blues,70,15,15,100
classical,70,15,15,100
country,70,15,15,100
disco,70,15,15,100
hiphop,70,15,15,100
jazz,69,15,15,99
metal,70,15,15,100
pop,70,15,15,100
reggae,70,15,15,100


In [7]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

feature_df.to_csv(
    FEATURE_FILE,
    index=False,
)

error_df.to_csv(
    ERROR_FILE,
    index=False,
)

if CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()

print("Feature file saved:")
print(FEATURE_FILE)

print("\nError report saved:")
print(ERROR_FILE)

print("\nFeature file exists:", FEATURE_FILE.exists())
print("Saved feature rows:", len(feature_df))

Feature file saved:
D:\Projects\music-genre-classification-AI\data\processed\audio_features.csv

Error report saved:
D:\Projects\music-genre-classification-AI\data\processed\feature_extraction_errors.csv

Feature file exists: True
Saved feature rows: 999


In [8]:
saved_feature_df = pd.read_csv(
    FEATURE_FILE
)

saved_metadata_columns = [
    "file_path",
    "filename",
    "genre",
    "split",
]

saved_feature_columns = [
    column
    for column in saved_feature_df.columns
    if column not in saved_metadata_columns
]

print("Saved rows:", len(saved_feature_df))
print("Saved columns:", len(saved_feature_df.columns))
print(
    "Numerical feature columns:",
    len(saved_feature_columns),
)

print(
    "Missing numerical values:",
    saved_feature_df[
        saved_feature_columns
    ].isna().sum().sum(),
)

display(saved_feature_df.head())

Saved rows: 999
Saved columns: 94
Numerical feature columns: 90
Missing numerical values: 0


,file_path,filename,genre,split,loaded_duration_seconds,tempo_bpm,mfcc_01_mean,mfcc_01_std,mfcc_02_mean,mfcc_02_std,...,spectral_contrast_03_mean,spectral_contrast_03_std,spectral_contrast_04_mean,spectral_contrast_04_std,spectral_contrast_05_mean,spectral_contrast_05_std,spectral_contrast_06_mean,spectral_contrast_06_std,spectral_contrast_07_mean,spectral_contrast_07_std
0,data\raw\genres_original\country\country.00061...,country.00061.wav,country,train,30.0,123.046875,-482.063802,68.180736,135.860073,28.998235,...,23.540774,4.545851,21.173070,3.812318,21.600273,3.830388,19.884863,3.602342,37.326387,4.477069
1,data\raw\genres_original\classical\classical.0...,classical.00046.wav,classical,validation,30.0,129.199219,-373.008461,23.495534,118.423067,19.340320,...,18.463920,3.502241,19.849700,2.520823,20.713156,2.359407,20.558529,2.448312,21.254693,2.730834
2,data\raw\genres_original\country\country.00005...,country.00005.wav,country,train,30.0,129.199219,-393.541695,27.496700,62.388383,15.932810,...,19.019879,4.381183,19.061713,3.924187,19.638747,3.195603,17.685056,2.671622,17.485123,2.074316
3,data\raw\genres_original\disco\disco.00049.wav,disco.00049.wav,disco,train,30.0,123.046875,-437.577377,60.125883,98.671647,30.238519,...,17.128404,4.040533,16.994300,3.413414,18.271373,3.387091,18.764664,3.561410,42.711696,6.376284
4,data\raw\genres_original\hiphop\hiphop.00022.wav,hiphop.00022.wav,hiphop,validation,30.0,99.384014,-501.525654,73.550980,110.210732,29.239844,...,18.053938,4.430873,16.798288,3.913257,16.911758,3.270236,16.258567,2.861101,40.968474,4.152934
